# Colab ⇄ Computer Bridge

Run the cells in order. This notebook will:

1. Make `server.py` available (clone from `REPO_URL` if you are running outside the repo).
2. Start a tiny HTTP server on `http://127.0.0.1:8787`.
3. Install `cloudflared` if missing and open a free Cloudflare HTTPS tunnel.
4. Print a public URL — copy it and paste it to the AI.

> ⚠️ **Security**: the tunnel exposes a *shell-executing* server on the public internet.
> Anyone with the URL can run commands as this user. Do NOT leave it running unattended,
> and strongly prefer setting `BRIDGE_TOKEN` in the config cell below.

In [ ]:
# ============ CONFIGURATION ============
# Running outside this repo (e.g. on Google Colab)? Put a cloneable URL here.
REPO_URL = "https://github.com/ASAYMAN69/colab.git"   # clone source for the server

SERVER_PORT = 8787       # local port the server listens on

# Optional bearer token. Empty = NO AUTH (anyone with the URL can act).
BRIDGE_TOKEN = ""
# ========================================

In [ ]:
import glob, os, shutil

HERE = os.getcwd()
SERVER_PY = os.path.join(HERE, "server.py")

if not os.path.exists(SERVER_PY):
    if REPO_URL:
        get_ipython().system(f"git clone --depth 1 {REPO_URL} _src")
        matches = glob.glob(os.path.join("_src", "**", "server.py"), recursive=True)
        if not matches:
            raise SystemExit("server.py not found inside the cloned repo")
        shutil.copy(matches[0], HERE)
        print("cloned repo and copied server.py")
    else:
        raise SystemExit(
            "server.py not found and REPO_URL is empty.\n"
            "Either run this notebook from inside the repository, "
            "or set REPO_URL to a cloneable copy."
        )

print("server.py ready at", SERVER_PY)

In [ ]:
import json, os, subprocess, sys, time, urllib.request

env = dict(os.environ)
if BRIDGE_TOKEN:
    env["BRIDGE_TOKEN"] = BRIDGE_TOKEN

proc = subprocess.Popen(
    [sys.executable, "server.py", "--port", str(SERVER_PORT)],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

ready = False
for _ in range(50):
    try:
        with urllib.request.urlopen(
            f"http://127.0.0.1:{SERVER_PORT}/status", timeout=1
        ) as r:
            print("server online:", json.load(r))
            ready = True
            break
    except Exception:
        time.sleep(0.2)

if not ready:
    out = proc.stdout.read() if proc.stdout else ""
    raise SystemExit("server failed to start:\n" + out)

In [ ]:
import os, platform, shutil, stat, urllib.request

CF = shutil.which("cloudflared") or os.path.expanduser("~/.local/bin/cloudflared")

if not os.path.exists(CF):
    machine = platform.machine()
    tag = {
        "x86_64": "amd64", "AMD64": "amd64",
        "aarch64": "arm64", "arm64": "arm64",
    }.get(machine)
    if not tag:
        raise SystemExit("unsupported CPU architecture: " + machine)
    os.makedirs(os.path.dirname(CF), exist_ok=True)
    url = ("https://github.com/cloudflare/cloudflared/releases/latest/download/"
           f"cloudflared-linux-{tag}")
    print("downloading cloudflared:", url)
    urllib.request.urlretrieve(url, CF)
    os.chmod(CF, os.stat(CF).st_mode | stat.S_IXUSR)
    print("installed cloudflared at", CF)

print("cloudflared ready:", CF)

In [ ]:
import re, subprocess, threading, time

tunnel = subprocess.Popen(
    [CF, "tunnel", "--url", f"http://127.0.0.1:{SERVER_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

URL_RE = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
url, logs = None, []

def reader():
    global url
    for line in iter(tunnel.stdout.readline, ""):
        logs.append(line.rstrip())
        m = URL_RE.search(line)
        if m:
            url = m.group(0)

threading.Thread(target=reader, daemon=True).start()

deadline = time.time() + 90
while time.time() < deadline and url is None:
    time.sleep(0.5)

if url:
    with open("cloudflare_url.txt", "w") as fh:
        fh.write(url + "\n")
    print()
    print("=" * 68)
    print("PUBLIC URL -> " + url)
    print("=" * 68)
    print("Copy this URL and paste it to the AI.")
    print("The tunnel stays open while this process runs. Stop the notebook when done.")
else:
    print("No Cloudflare URL within 90s. Last tunnel logs:")
    print("\n".join(logs[-15:]))
    print("Tip: quick tunnels are sometimes rate-limited - wait a minute and rerun this cell.")

In [ ]:
# Stop the background processes (server + tunnel) so the public URL closes.
for p, name in ((locals().get("proc"), "server"), (locals().get("tunnel"), "tunnel")):
    if p is not None and p.poll() is None:
        p.terminate()
        print("stopped", name)
print("cleanup done")